In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data = np.load("Data/ZH_radar_dataset.npy")  # shape (288, 14, 360, 240)
ZH_max = np.max(data)
data = data / ZH_max  # normalize to 0-1

In [ ]:
seq_len = 10
num_total = data.shape[0]
num_train = int(num_total * 0.6)  # first 60%

train_data = data[:num_train]  # e.g., [:172]
test_data = data[num_train - seq_len:]  # overlap by seq_len for continuity


In [ ]:
class RadarDataset(Dataset):
    def __init__(self, data, seq_len=10):
        self.X, self.y = [], []
        for i in range(len(data) - seq_len):
            self.X.append(data[i:i+seq_len])      # shape: [10, 14, 360, 240]
            self.y.append(data[i+seq_len])        # shape: [14, 360, 240]
        self.X = np.array(self.X)
        self.y = np.array(self.y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


In [ ]:
train_dataset = RadarDataset(train_data, seq_len=seq_len)
test_dataset = RadarDataset(test_data, seq_len=seq_len)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)


In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        conv_out = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.chunk(conv_out, 4, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class ConvLSTMModel(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=64, output_dim=14):
        super().__init__()
        self.lstm = ConvLSTMCell(input_dim, hidden_dim)
        self.out_conv = nn.Conv2d(hidden_dim, output_dim, kernel_size=1)

    def forward(self, x_seq):  # x_seq: [B, T, C, H, W]
        B, T, C, H, W = x_seq.size()
        h = torch.zeros(B, 64, H, W).to(x_seq.device)
        c = torch.zeros(B, 64, H, W).to(x_seq.device)
        for t in range(T):
            h, c = self.lstm(x_seq[:, t], h, c)
        return self.out_conv(h)


In [ ]:
# class WeightedMSELoss(nn.Module):
#     def __init__(self, alpha=10.0):  # increase this if you want to put more weight on high reflectivity values
#         super().__init__()
#         self.alpha = alpha

#     def forward(self, pred, target):
#         weight = 1 + self.alpha * (target > 30).float()  # We put alpha times more weight on reflectivity values over 30
#         return torch.mean(weight * (pred - target) ** 2)
    


threshold_norm = 30.0 / ZH_max  # Normalize the 30 threshold

# We put alpha times more weight on reflectivity values over 30
class WeightedMSELoss(nn.Module):
    def __init__(self, alpha=10.0, threshold=threshold_norm): # increase alpha if you want to put more weight on high reflectivity values
        super().__init__()
        self.alpha = alpha
        self.threshold = threshold

    def forward(self, pred, target):
        weight = 1 + self.alpha * (target > self.threshold).float()
        return torch.mean(weight * (pred - target) ** 2)



In [ ]:
model = ConvLSTMModel().to(device)
criterion = nn.MSELoss()
#criterion = WeightedMSELoss(alpha = 30.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
    for batch_x, batch_y in loop:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} finished - Total Train Loss: {total_loss:.4f}")


In [ ]:
torch.save(model.state_dict(), "checkpoints/mse_model.pth")

In [ ]:
model.eval()
test_loss = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        test_loss += loss.item()

print(f"🧪 Final Test Loss: {test_loss:.4f}")


In [ ]:
model.eval()
pred_list = []

with torch.no_grad():
    for i in range(len(test_data) - 10):
        input_seq = torch.tensor(test_data[i:i+10], dtype=torch.float32).unsqueeze(0).to(device)  # [1, 10, 14, 360, 240]
        pred = model(input_seq)  # [1, 14, 360, 240]
        pred = pred.squeeze(0).cpu().numpy()  # [14, 360, 240]
        pred_list.append(pred)

pred_data = np.stack(pred_list)  # shape: [116, 14, 360, 240]


In [ ]:
y_test_data = test_data[10:]  # shape becomes (116, 14, 360, 240)


In [ ]:
predicted_data = np.transpose(pred_data, (0, 2, 3, 1))  # (288, 360, 240, 14)
true_data = np.transpose(y_test_data, (0, 2, 3, 1))  # (288, 360, 240, 14)
predicted_data.shape, true_data.shape

In [ ]:
predictions= predicted_data * ZH_max
true = true_data * ZH_max
pred_max_cappi = np.max(predictions, axis=-1)
true_max_cappi = np.max(true, axis=-1)

In [ ]:
true_max_cappi

In [ ]:
pred_max_cappi

In [ ]:
from storm_utils import animate_storms
from IPython.display import HTML, display
ani = animate_storms(true_max_cappi)
display(HTML(ani.to_jshtml()))

In [ ]:
ani = animate_storms(pred_max_cappi)
display(HTML(ani.to_jshtml()))

# Train model

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Use Mac GPU if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Evaluate model prediction

In [ ]:
import numpy as np
data = np.load("Data/ZH_radar_dataset.npy") 
# We are actually just interested in storm formation in our testing set. Because this is where we will evalueate if our model is able to predict storm formation.
train_size = int(len(data) * 0.6)
#train_data = data[:train_size]
test_set = data[train_size:]
test_set.shape

In [ ]:
#maxcappi of testdata
test_data = np.max(test_set, axis=1)
test_data.shape

## Storm progression over time

In [ ]:
from storm_utils import animate_storms
from IPython.display import HTML, display
# import matplotlib as mpl
# mpl.rcParams['animation.embed_limit'] = 50  # in MB, can if there's a lot of data to display
ani = animate_storms(test_data)
display(HTML(ani.to_jshtml()))

## Detects storms and returns their count and coordinates at each time step

In [ ]:
from storm_utils import detect_storms
storm_data = detect_storms(test_data)
storm_data

## Detects new storm formations: frames, count of newly formed storms, and their coordinates.

In [ ]:
from storm_utils import detect_new_storm_formations
new_storms = detect_new_storm_formations(test_data)
new_storms

## Progression of *newly* formed storms over time

In [ ]:
from storm_utils import animate_new_storms
new_storms = detect_new_storm_formations(test_data)
ani = animate_new_storms(test_data, new_storms)
display(HTML(ani.to_jshtml()))


TODO: plot the new storms of predictions and test data to compare. And create some sort of function that calculates the accuracy of newly formed predictions